In [ ]:
import pandas as pd 
import numpy as np


In [ ]:
car=pd.read_csv("quikr_car.csv")

In [ ]:
car.head()

In [ ]:
car.shape

In [ ]:
car.info()

In [ ]:
car['year'].unique()

In [ ]:
car['Price'].unique()

In [ ]:
car['kms_driven'].unique()

In [ ]:
car["fuel_type"].unique()

## Errors found in dataset

- flaws in data of yeaar 
- convert year data from obj to int
- price data comma removal 
- remove ask for prediction 
- convert obj into integer
- IN kms_driven remove kms and commas 
- convert obj into integer 
- there is nan value
- In fuel_typee there is a nan value
- keep only first 3 words in name 


## cleaning

In [ ]:
backup=car.copy()

In [ ]:
car["year"].str.isnumeric()

In [ ]:
car=car[car["year"].str.isnumeric()]

In [ ]:
car['year'] = car['year'].astype(int)


In [ ]:
car['year']

In [ ]:
car=car[car['Price']!="Ask For Price"]

In [ ]:
car

In [ ]:
car['Price'] = car['Price'].str.replace(',','')


In [ ]:
car['Price']=car['Price'].astype(int)

In [ ]:
car.info()

In [ ]:
car['kms_driven']=car['kms_driven'].str.split(" ").str.get(0)

In [ ]:
car['kms_driven']=car['kms_driven'].str.replace(",","") 

In [ ]:
car['kms_driven']

In [ ]:
car=car[car["kms_driven"].str.isnumeric()]

In [ ]:
car['kms_driven']=car['kms_driven'].astype(int)

In [ ]:
car

In [ ]:
car[car['fuel_type'].isna()]

In [ ]:
car=car[~car['fuel_type'].isna()]

In [ ]:
car.info()

In [ ]:
car['name']=car['name'].str.split(' ').str.slice(0,3).str.join(' ')

In [ ]:
car

In [ ]:
car=car.reset_index(drop=True)

In [ ]:
car


In [ ]:
car.describe()

In [ ]:
car[car['Price']>6e6]

In [ ]:
car=car[car['Price']<6e6].reset_index(drop=True)

In [ ]:
car

In [ ]:
car.to_csv("cleaned car data.csv")

## Model

In [ ]:
x=car.drop(columns='Price')
y=car['Price']

In [ ]:
x


In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test , y_train, y_test = train_test_split(x,y,test_size=0.2)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

In [ ]:
ohe=OneHotEncoder()
ohe.fit(x[['name','company','fuel_type']])

In [ ]:
column_trans=make_column_transformer((OneHotEncoder(categories=ohe.categories_),['name','company','fuel_type']),remainder='passthrough')

In [ ]:
lr=LinearRegression()

In [ ]:
pipe= make_pipeline(column_trans,lr)

In [ ]:
pipe.fit(x_train, y_train)

In [ ]:
y_pred=pipe.predict(x_test)

In [ ]:
r2_score(y_test,y_pred)

In [ ]:
scores=[]
for i in range(1000):
    x_train, x_test , y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=i)
    lr=LinearRegression()
    pipe= make_pipeline(column_trans,lr)
    pipe.fit(x_train, y_train)
    y_pred=pipe.predict(x_test)
    scores.append(r2_score(y_test,y_pred))

In [ ]:
np.argmax(scores)

In [ ]:
scores[np.argmax(scores)]

In [ ]:
x_train, x_test , y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=np.argmax(scores))
lr=LinearRegression()
pipe= make_pipeline(column_trans,lr)
pipe.fit(x_train, y_train)
y_pred=pipe.predict(x_test)
r2_score(y_test,y_pred)

In [ ]:
import pickle

In [ ]:
pickle.dump(pipe,open('LinearRegressionModel.pkl','wb'))

In [ ]:
pipe.predict(pd.DataFrame([['Maruti Suzuki Swift','Maruti',2016,100000,'Petrol']],columns=['name','company','year','kms_driven','fuel_type']))

In [ ]:
car.head()